# 05 — Fluxo completo: fusão multimodal e alerta

Este notebook roda o sistema inteiro para os três pacientes de demonstração (`data/pacientes_demo.json`):

```
vídeo ──► MediaPipe + YOLOv8 ──► desvios de execução, movimentação ─┐
voz ────► características acústicas ──► classificador ──────────────┤
consulta ► Azure Speech to Text ──► Azure Text Analytics + termos ──┼─► risco ─► alerta
prescrições ──► regras ─────────────────────────────────────────────┤           (texto redigido
sinais vitais (tempo real) ──► limites clínicos + Isolation Forest ─┘            pelo Azure OpenAI)
```

**Regra de fusão** (`src/fusao/risco.py`):

1. De cada fonte vale só o evento mais grave (severidade 1 a 3).
2. A pontuação é a soma dessas severidades.
3. **Alto:** pontuação 5 ou mais, ou qualquer evento grave. **Médio:** 2 a 4. **Baixo:** 0 ou 1.

Um alerta é emitido quando o risco é médio ou alto, e de novo apenas se o nível **subir**.
A decisão é sempre das regras; o GPT só redige o texto do relatório.

In [1]:
import os
import sys
from pathlib import Path

RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

import json
import warnings

import pandas as pd

import monitor
from src.fusao.risco import calcular_risco
from src.modelos import Evento

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)

ALERTAS = Path("results/alertas.jsonl")
ALERTAS.unlink(missing_ok=True)  # começa com o registro de alertas vazio

pd.DataFrame(json.loads(Path("data/pacientes_demo.json").read_text(encoding="utf-8"))).T

,descricao,video,voz,consulta,sinais_vitais
P001,paciente estável,data/amostras/fisio_correta.mp4,data/amostras/voz_saudavel.wav,data/amostras/consulta_01_rotina.wav,data/amostras/sinais_vitais_normal.csv
P002,paciente com queixas leves e ajuste de medicação; piora durante o monitoramento,data/amostras/fisio_correta.mp4,data/amostras/voz_saudavel.wav,data/amostras/consulta_02_cansaco.wav,data/amostras/sinais_vitais_demo.csv
P003,paciente com vários sinais de piora já antes do monitoramento,data/amostras/fisio_incorreta.mp4,data/amostras/voz_patologica.wav,data/amostras/consulta_03_falta_de_ar.wav,data/amostras/sinais_vitais_demo.csv


## 1. A regra de fusão em exemplos

In [2]:
exemplos = {
    "nenhum evento": [],
    "só voz alterada (leve)": [Evento("voz", "", 1, 0, "")],
    "vídeo moderado + texto moderado": [Evento("video", "", 2, 0, ""), Evento("texto", "", 2, 0, "")],
    "três fontes moderadas": [Evento("video", "", 2, 0, ""), Evento("texto", "", 2, 0, ""), Evento("prescricao", "", 2, 0, "")],
    "um único evento grave": [Evento("sinais_vitais", "", 3, 0, "")],
    "falha de sensor + voz leve": [Evento("sensor", "", 1, 0, ""), Evento("voz", "", 1, 0, "")],
}
pd.DataFrame([
    {"situação": nome, "pontuação": (r := calcular_risco(eventos)).pontuacao, "nível": r.nivel, "alerta?": "sim" if r.nivel != "baixo" else "não"}
    for nome, eventos in exemplos.items()
])

,situação,pontuação,nível,alerta?
0,nenhum evento,0,baixo,não
1,só voz alterada (leve),1,baixo,não
2,vídeo moderado + texto moderado,4,medio,sim
3,três fontes moderadas,6,alto,sim
4,um único evento grave,3,alto,sim
5,falha de sensor + voz leve,1,baixo,não


## 2. Paciente P001: estável

Nenhuma fonte gera evento. O risco termina baixo e **nenhum alerta é emitido**.

In [3]:
monitor.monitorar("P001", velocidade=0, usar_gpt=True)


##############################################################################
# Paciente P001: paciente estável
##############################################################################

[1/5 Vídeo de fisioterapia] processando...


    vídeo anotado: results/P001_video_anotado.mp4

--- 1/5 Vídeo de fisioterapia ---
    nenhum evento

[2/5 Voz] processando...



--- 2/5 Voz ---
    nenhum evento

[3/5 Áudio da consulta] processando...


    transcrição: "Bom dia, doutora. Estou me sentindo bem esta semana. Tenho dormido a noite inteira e voltei a fazer minhas caminhadas pela manhã. O apetite está normal e estou tomando os remédios nos horários certos. Não tive nenhum problema desde a última consulta."



--- 3/5 Áudio da consulta ---
    nenhum evento

[4/5 Prescrições] processando...

--- 4/5 Prescrições ---
    nenhum evento

>>> Risco antes dos sinais vitais: BAIXO (pontuação 0, fontes: nenhuma)

[5/5 Sinais vitais em tempo real] 0x a velocidade real
    t=   0s  FC 83  SpO2 95  FR 17  PAS 121


    t=  30s  FC 84  SpO2 95  FR 17  PAS 121


... (12 outputs left out)

    t= 420s  FC 84  SpO2 95  FR 17  PAS 128


    t= 450s  FC 84  SpO2 95  FR 17  PAS 122


    t= 480s  FC 84  SpO2 95  FR 17  PAS 115

##############################################################################
# Fim do monitoramento de P001: risco BAIXO (pontuação 0)
# Eventos: 0 | fontes: nenhuma
# Alertas registrados em results/alertas.jsonl
##############################################################################



0

## 3. Paciente P002: piora durante o monitoramento

Queixas leves na consulta e um aumento de dose dão **risco médio** (primeiro alerta).
Durante o monitoramento, a oxigenação cai abaixo do limite clínico e o risco sobe para **alto** (segundo alerta, no mesmo instante).

In [4]:
monitor.monitorar("P002", velocidade=0, usar_gpt=True)


##############################################################################
# Paciente P002: paciente com queixas leves e ajuste de medicação; piora durante o monitoramento
##############################################################################

[1/5 Vídeo de fisioterapia] processando...


    vídeo anotado: results/P002_video_anotado.mp4

--- 1/5 Vídeo de fisioterapia ---
    nenhum evento

[2/5 Voz] processando...



--- 2/5 Voz ---
    nenhum evento

[3/5 Áudio da consulta] processando...


    transcrição: "Boa tarde, doutor. Nos últimos dias, tenho sentido muito cansaço. Qualquer atividade simples como arrumar a cama, já me deixa com fadiga. Também estou com insônia, acordo várias vezes durante a noite. Fora isso, não sinto dor, mas estou sem disposição para nada."



--- 3/5 Áudio da consulta ---
    [severidade 1] Paciente mencionou "cansaço" na consulta
    [severidade 1] Paciente mencionou "fadiga" na consulta
    [severidade 1] Paciente mencionou "insônia" na consulta
    [severidade 1] Sentimento negativo na fala do paciente (72%)

[4/5 Prescrições] processando...

--- 4/5 Prescrições ---
    [severidade 2] Paciente P002: dose de furosemida passou de 40 mg para 80 mg (100% de variação)

>>> Risco antes dos sinais vitais: MEDIO (pontuação 3, fontes: prescricao, texto)



  ALERTA PARA A EQUIPE MÉDICA | paciente P002 | risco MÉDIO | 2026-11-15T17:54:54
Paciente P002 apresenta risco MÉDIO. Foi detectada uma alteração significativa na prescrição, com aumento da dose de furosemida de 40 mg para 80 mg, representando uma variação de 100%. Além disso, o paciente relatou sintomas como cansaço, fadiga e insônia, e foi identificado um sentimento negativo em sua fala. Recomenda-se que a equipe médica avalie o paciente para monitorar esses sinais e ajustar o manejo conforme necessário.


[5/5 Sinais vitais em tempo real] 0x a velocidade real
    t=   0s  FC 90  SpO2 98  FR 18  PAS 115


... (16 outputs left out)

    t= 420s  FC 92  SpO2 98  FR 20  PAS 109


    t= 450s  FC 90  SpO2 98  FR 18  PAS 122


    t= 480s  FC 89  SpO2 98  FR 19  PAS 122

##############################################################################
# Fim do monitoramento de P002: risco ALTO (pontuação 6)
# Eventos: 8 | fontes: prescricao, sinais_vitais, texto
# Alertas registrados em results/alertas.jsonl
##############################################################################



0

## 4. Paciente P003: vários sinais de piora

Vídeo, voz, consulta e prescrições já somam **risco alto** antes mesmo dos sinais vitais.
Os eventos de sinais vitais são registrados, mas não geram novo alerta: o nível já era o máximo.

In [5]:
monitor.monitorar("P003", velocidade=0, usar_gpt=True)


##############################################################################
# Paciente P003: paciente com vários sinais de piora já antes do monitoramento
##############################################################################

[1/5 Vídeo de fisioterapia] processando...


    vídeo anotado: results/P003_video_anotado.mp4

--- 1/5 Vídeo de fisioterapia ---
    [severidade 2] Amplitude de ombro_dir abaixo do esperado: pico de 145° (esperado entre 154° e 185°)

[2/5 Voz] processando...



--- 2/5 Voz ---
    [severidade 1] Características da voz compatíveis com alteração vocal (92%)

[3/5 Áudio da consulta] processando...


    transcrição: "Doutora, estou preocupado desde ontem, sinto falta de ar até quando estou parado. Hoje de manhã tive uma dor no peito muito forte, que durou alguns minutos. Também senti palpitação e precisei me sentar. Estou com medo de que seja algo grave."



--- 3/5 Áudio da consulta ---
    [severidade 3] Paciente mencionou "falta de ar" na consulta
    [severidade 3] Paciente mencionou "dor no peito" na consulta
    [severidade 2] Paciente mencionou "palpitação" na consulta
    [severidade 1] Sentimento negativo na fala do paciente (100%)

[4/5 Prescrições] processando...

--- 4/5 Prescrições ---
    [severidade 3] Paciente P003: dose de morfina passou de 5 mg para 15 mg (200% de variação)
    [severidade 2] Paciente P003: medicamentos novos iniciados em menos de 24 h: amiodarona e varfarina

>>> Risco antes dos sinais vitais: ALTO (pontuação 9, fontes: prescricao, texto, video, voz)



  ALERTA PARA A EQUIPE MÉDICA | paciente P003 | risco ALTO | 2026-11-15T17:55:32
Paciente P003 apresenta risco ALTO (pontuação 9). Foram detectados sintomas graves como falta de ar e dor no peito, além de aumento significativo na dose de morfina (de 5 mg para 15 mg). Também foram iniciados medicamentos novos, amiodarona e varfarina, há menos de 24 horas. Sintomas moderados incluem palpitação e amplitude reduzida do ombro direito. Alterações leves foram observadas na voz e no sentimento expresso pelo paciente. Recomenda-se avaliação imediata pela equipe médica para monitoramento e conduta adequada.


[5/5 Sinais vitais em tempo real] 0x a velocidade real


... (16 outputs left out)

    t= 420s  FC 92  SpO2 98  FR 20  PAS 109


    t= 450s  FC 90  SpO2 98  FR 18  PAS 122


    t= 480s  FC 89  SpO2 98  FR 19  PAS 122

##############################################################################
# Fim do monitoramento de P003: risco ALTO (pontuação 12)
# Eventos: 11 | fontes: prescricao, sinais_vitais, texto, video, voz
# Alertas registrados em results/alertas.jsonl
##############################################################################



0

## 5. Registro dos alertas

Cada alerta fica em uma linha de `results/alertas.jsonl`, com os eventos que o justificam.

In [6]:
alertas = [json.loads(linha) for linha in ALERTAS.read_text(encoding="utf-8").strip().split("\n")]
pd.DataFrame([
    {"paciente": a["paciente_id"], "nível": a["nivel"], "pontuação": a["pontuacao"],
     "fontes": ", ".join(a["fontes"]), "eventos": len(a["eventos"]), "relatório (início)": a["relatorio"][:110] + "..."}
    for a in alertas
])

,paciente,nível,pontuação,fontes,eventos,relatório (início)
0,P002,medio,3,"prescricao, texto",5,"Paciente P002 apresenta risco MÉDIO. Foi detectada uma alteração significativa na prescrição, com aumento da d..."
1,P002,alto,6,"prescricao, sinais_vitais, texto",6,"Paciente P002 apresenta risco ALTO (pontuação 6). Foram detectados sinais vitais fora do padrão, com SpO2 em 8..."
2,P003,alto,9,"prescricao, texto, video, voz",8,Paciente P003 apresenta risco ALTO (pontuação 9). Foram detectados sintomas graves como falta de ar e dor no p...


In [7]:
print(json.dumps(alertas[1], ensure_ascii=False, indent=2))

{
  "paciente_id": "P002",
  "data_hora": "2026-11-15T17:55:03",
  "nivel": "alto",
  "pontuacao": 6,
  "fontes": [
    "prescricao",
    "sinais_vitais",
    "texto"
  ],
  "eventos": [
    {
      "fonte": "sinais_vitais",
      "tipo": "limite_clinico",
      "severidade": 3,
      "instante": 118.0,
      "descricao": "Sinais vitais fora do padrão (SPO2). Leituras: FC 89 bpm, SPO2 84 %, FR 19 rpm, PAS 120 mmHg"
    },
    {
      "fonte": "prescricao",
      "tipo": "mudanca_brusca_de_dose",
      "severidade": 2,
      "instante": 172800.0,
      "descricao": "Paciente P002: dose de furosemida passou de 40 mg para 80 mg (100% de variação)"
    },
    {
      "fonte": "texto",
      "tipo": "termo_critico",
      "severidade": 1,
      "instante": 0.0,
      "descricao": "Paciente mencionou \"cansaço\" na consulta"
    },
    {
      "fonte": "texto",
      "tipo": "termo_critico",
      "severidade": 1,
      "instante": 0.0,
      "descricao": "Paciente mencionou \"fadiga\" na co